# Joseph's Indicators — v10
## Industrial–Logistics Coupling in Otay Mesa | San Diego Border Zone

**Changes from v9:**
- CRS changed from EPSG:2230 (US survey feet) to **EPSG:3310** (California Albers, meters) — matches Christian's code exactly.
- Subarea assignment now uses **Community_Plan_SD.geojson** spatial join instead of a hardcoded bounding box — matches Christian's planning district approach.
- **Four study areas** defined: Otay Mesa, Kearny Mesa, Miramar, Sorrento Valley (+ Other).
- Buffer distances converted from feet to **meters** (402m, 805m, 1609m = 0.25/0.50/1.00 mi).
- `dist_to_freight_road` now uses `gpd.sjoin_nearest` — matches Christian's method.
- Sorrento Valley mapped to MIRA MESA planning district (no official Sorrento Valley district exists).

**Run after:** `industrial_businesses_core_geocoded.csv` and `Community_Plan_SD.geojson` are in the same directory.
**Run before:** `joseph_visualizations_v8.ipynb`


## A. Setup

In [1]:
import requests
import geopandas as gpd
import pandas as pd
import numpy as np
from shapely.geometry import Point
from io import BytesIO
import warnings
warnings.filterwarnings("ignore")

#  CRS — matches Christian's code exactly
CRS_GEO  = "EPSG:4326"    # geographic lat/lon
CRS_PROJ = "EPSG:3310"    # California Albers (meters) — same as Christian

METERS_PER_MILE = 1609.34
FT_PER_MILE     = 5280.0

#  Buffer distances in METERS (0.25 / 0.50 / 1.00 mi)
BUFFER_DISTANCES_M = {
    "buffer_quarter_mi": 402,
    "buffer_half_mi":    805,
    "buffer_one_mi":     1609,
}

#  Input files
INPUT_CSV           = "industrial_businesses_core_geocoded.csv"
COMMUNITY_PLAN_FILE = "Community_Plan_SD.geojson"

#  Study area mapping: official CPNAME -> study area label
#  Otay Mesa     = OTAY MESA + OTAY MESA-NESTOR  (matches Christian str.contains)
#  Kearny Mesa   = KEARNY MESA
#  Miramar       = MIRAMAR RANCH NORTH + SCRIPPS MIRAMAR RANCH (Canyu circle covers both)
#  Sorrento Valley = MIRA MESA (no official Sorrento Valley district; closest match)
STUDY_AREA_MAP = {
    "OTAY MESA":             "Otay Mesa",
    "OTAY MESA-NESTOR":      "Otay Mesa",
    "KEARNY MESA":           "Kearny Mesa",
    "MIRAMAR RANCH NORTH":   "Miramar",
    "SCRIPPS MIRAMAR RANCH": "Miramar",
    "MIRA MESA":             "Sorrento Valley",
}
BORDER_AREA      = "Otay Mesa"
COMPARISON_AREAS = ["Kearny Mesa", "Miramar", "Sorrento Valley"]

print("Config ready.")
print(f"CRS: {CRS_PROJ}")
print(f"Buffer distances (m): {BUFFER_DISTANCES_M}")



Config ready.
CRS: EPSG:3310
Buffer distances (m): {'buffer_quarter_mi': 402, 'buffer_half_mi': 805, 'buffer_one_mi': 1609}


## B. Load Industrial Business Points

In [2]:
df = pd.read_csv(INPUT_CSV, dtype=str)
print(f"Loaded {len(df)} rows")
print("Columns:", df.columns.tolist())
df.head(3)



Loaded 3067 rows
Columns: ['business_acctnum', 'dba_name', 'ownership_type', 'address', 'city', 'state', 'zip', 'naics', 'activity_desc', 'industry_group', 'definition', 'full_address', 'matched_address', 'geocode_score', 'longitude', 'latitude']


,business_acctnum,dba_name,ownership_type,address,city,state,zip,naics,activity_desc,industry_group,definition,full_address,matched_address,geocode_score,longitude,latitude
0,2021003855,098 ENTERPRISES,CORP,3412 LITTLE FLOWER ST,SAN DIEGO,CA,92104-5225,422,"WHOLESALE TRADE, NONDURABLE GOODS",Wholesale Trade,Core,"3412 LITTLE FLOWER ST , SAN DIEGO, CA 92104-5225","2038 Corte del Nogal, Suite 134, Carlsbad, Cal...",100.0,-117.279002413749,33.118762156516
1,1998011168,1 SPIRIT,CORP,3830 VALLEY CENTRE DR 705-903,SAN DIEGO,CA,92130-3320,42282,WINE & DISTILLED ALCOHOLIC BEVERAGE WHSLE,Wholesale Trade,Core,"3830 VALLEY CENTRE DR 705-903 , SAN DIEGO, CA ...","10792 Roselle St, San Diego, California, 92121",100.0,-117.222966541174,32.899788288819
2,2003009618,1-800-GOTJUNK OF SAN DIEGO,LLC,2038 CORTE DEL NOGAL 134,CARLSBAD,CA,92011-1478,488999,ALL OTHER TRANSPORTATION SUPPORT ACTIVITIES,Transportation and Warehousing,Core,"2038 CORTE DEL NOGAL 134 , CARLSBAD, CA 92011-...","3412 Little Flower St, San Diego, California, ...",100.0,-117.119854212329,32.736866931774


In [3]:
df["longitude"] = pd.to_numeric(df["longitude"], errors="coerce")
df["latitude"]  = pd.to_numeric(df["latitude"],  errors="coerce")
df = df.dropna(subset=["longitude", "latitude"]).reset_index(drop=True)
print(f"Rows with valid coordinates: {len(df)}")

businesses = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df["longitude"], df["latitude"]),
    crs=CRS_GEO
).to_crs(CRS_PROJ)

print(f"GeoDataFrame CRS: {businesses.crs}")
businesses[["dba_name", "industry_group", "city", "latitude", "longitude"]].head(5)



Rows with valid coordinates: 3067
GeoDataFrame CRS: EPSG:3310


,dba_name,industry_group,city,latitude,longitude
0,098 ENTERPRISES,Wholesale Trade,SAN DIEGO,33.118762,-117.279002
1,1 SPIRIT,Wholesale Trade,SAN DIEGO,32.899788,-117.222967
2,1-800-GOTJUNK OF SAN DIEGO,Transportation and Warehousing,CARLSBAD,32.736867,-117.119854
3,100% SPEEDLAB LLC,Wholesale Trade,SAN DIEGO,32.745302,-117.198323
4,102 SCONE CO.,Manufacturing,LA JOLLA,32.937932,-117.230644


## B2. Load Community Planning Districts & Clip to SD

In [4]:
# Load Community Planning District boundaries
planning = gpd.read_file(COMMUNITY_PLAN_FILE)
if planning.crs is None:
    planning = planning.set_crs(CRS_GEO)
planning = planning.to_crs(CRS_PROJ)
print(f"Planning districts loaded: {len(planning)} features")
print("Columns:", planning.columns.tolist())
print("Sample CPNAME values:", sorted(planning["CPNAME"].unique())[:10])



Planning districts loaded: 61 features
Columns: ['OBJECTID', 'CPCODE', 'CPNAME', 'ACREAGE', 'Website', 'Shape_Length', 'Shape_Area', 'geometry']
Sample CPNAME values: ['BALBOA PARK', 'BARRIO LOGAN', 'BLACK MOUNTAIN RANCH', 'CARMEL MOUNTAIN RANCH', 'CARMEL VALLEY', 'CLAIREMONT MESA', 'COLLEGE AREA', 'DEL MAR MESA', 'DOWNTOWN', 'EAST ELLIOTT']


In [5]:
# Clip businesses to SD city planning area boundary (dissolve all districts)
sd_boundary = planning.geometry.unary_union
before = len(businesses)
businesses = businesses[businesses.geometry.within(sd_boundary)].copy().reset_index(drop=True)
after = len(businesses)
print(f"Businesses before clip: {before}")
print(f"Businesses after clip:  {after} (removed {before - after} outside SD city planning areas)")



Businesses before clip: 3067
Businesses after clip:  2500 (removed 567 outside SD city planning areas)


## C. Assign Study Area Subareas via Spatial Join

In [6]:
# Assign planning area via spatial join — matches Christian's approach exactly
businesses = gpd.sjoin(
    businesses,
    planning[["CPNAME", "geometry"]],
    how="left",
    predicate="within"
).drop(columns=["index_right"], errors="ignore")

businesses["CPNAME"] = businesses["CPNAME"].fillna("Outside City Planning Areas")

# Map CPNAME to study area label
businesses["subarea"] = businesses["CPNAME"].map(STUDY_AREA_MAP).fillna("Other")

print("\nSubarea counts:")
print(businesses["subarea"].value_counts())
print(f"\nTotal businesses: {len(businesses)}")




Subarea counts:
subarea
Other              1380
Otay Mesa           493
Sorrento Valley     369
Kearny Mesa         206
Miramar              52
Name: count, dtype: int64

Total businesses: 2500


## D. Fetch SANDAG Reference Layers

In [7]:
# Fetch SANDAG major freight roads — same URL as Christian
print("Fetching SANDAG Major Roads...")
major_roads = gpd.read_file(
    BytesIO(requests.get(
        "https://gis.sandag.org/FreightViewer/data/MajorRoads_20171002.json",
        timeout=60
    ).content)
).to_crs(CRS_PROJ)
print(f"Major roads: {len(major_roads)} features | CRS: {major_roads.crs}")

roads_dissolved = major_roads.dissolve().geometry.iloc[0]
print("Roads dissolved into single geometry.")



Fetching SANDAG Major Roads...
Major roads: 45 features | CRS: EPSG:3310
Roads dissolved into single geometry.


In [8]:
# Fetch POE points
print("Fetching SANDAG POE points...")
try:
    poe_pts = gpd.read_file(
        BytesIO(requests.get(
            "https://gis.sandag.org/FreightViewer/data/poe_points_update.json",
            timeout=60
        ).content)
    ).to_crs(CRS_PROJ)
    print(f"POE points: {len(poe_pts)} features")
    poe_source = "SANDAG FreightViewer"
except Exception as e:
    print(f"[fallback] SANDAG POE unavailable: {e}")
    poe_pts = gpd.GeoDataFrame(
        {"port_name": ["San Ysidro POE", "Otay Mesa POE"]},
        geometry=[Point(-117.0295, 32.5440), Point(-116.9447, 32.5726)],
        crs=CRS_GEO
    ).to_crs(CRS_PROJ)
    poe_source = "hardcoded fallback"
    print("Using hardcoded POE fallback.")

print(f"POE source: {poe_source}")
print("Columns:", poe_pts.columns.tolist())



Fetching SANDAG POE points...
POE points: 10 features
POE source: SANDAG FreightViewer
Columns: ['OBJECTID_1', 'Status', 'Existing', 'Future', 'Port_ID', 'port_name', 'url', 'wait_time', 'mode', 'geometry']


## E. Indicator 1 — Distance to Nearest Port of Entry

In [9]:
# Indicator 1 — Distance to nearest POE
# Distances computed in meters (CRS_PROJ = EPSG:3310), converted to miles

poe_name_col = next(
    (c for c in poe_pts.columns if "name" in c.lower() or "port" in c.lower()), None
)
print("POE name column:", poe_name_col)

if poe_name_col:
    poe_pts[poe_name_col] = poe_pts[poe_name_col].astype(str)
    sy_geom = poe_pts[
        poe_pts[poe_name_col].str.contains("Ysidro|ysidro|SAN YSIDRO", na=False, case=False)
    ].geometry.unary_union
    om_geom = poe_pts[
        poe_pts[poe_name_col].str.contains("Otay|otay|OTAY", na=False, case=False)
    ].geometry.unary_union
else:
    sy_geom, om_geom = None, None

if sy_geom is None:
    sy_geom = gpd.GeoDataFrame(
        geometry=[Point(-117.0295, 32.5440)], crs=CRS_GEO
    ).to_crs(CRS_PROJ).geometry.iloc[0]
    print("San Ysidro: using hardcoded coords")

if om_geom is None:
    om_geom = gpd.GeoDataFrame(
        geometry=[Point(-116.9447, 32.5726)], crs=CRS_GEO
    ).to_crs(CRS_PROJ).geometry.iloc[0]
    print("Otay Mesa POE: using hardcoded coords")

poe_union = poe_pts.geometry.unary_union

businesses["dist_nearest_poe_m"]  = businesses.geometry.distance(poe_union)
businesses["dist_nearest_poe_mi"] = businesses["dist_nearest_poe_m"] / METERS_PER_MILE

businesses["dist_san_ysidro_m"]  = businesses.geometry.distance(sy_geom)
businesses["dist_san_ysidro_mi"] = businesses["dist_san_ysidro_m"] / METERS_PER_MILE

businesses["dist_otay_poe_m"]  = businesses.geometry.distance(om_geom)
businesses["dist_otay_poe_mi"] = businesses["dist_otay_poe_m"] / METERS_PER_MILE

businesses["nearest_poe_name"] = np.where(
    businesses["dist_san_ysidro_m"] <= businesses["dist_otay_poe_m"],
    "San Ysidro", "Otay Mesa"
)

print("\n── Distance to Nearest POE (miles) by Subarea ──")
print(
    businesses.groupby("subarea")["dist_nearest_poe_mi"]
    .agg(n="count", mean="mean", median="median", std="std", min="min", max="max")
    .round(3)
)

print("\n── Which POE is Nearest, by Subarea ──")
print(
    businesses.groupby(["subarea", "nearest_poe_name"]).size()
    .reset_index(name="count")
)



POE name column: Port_ID

── Distance to Nearest POE (miles) by Subarea ──
                    n    mean  median    std     min     max
subarea                                                     
Kearny Mesa       206  20.545  20.745  0.667  18.278  21.953
Miramar            52  26.048  25.994  0.711  24.905  27.520
Otay Mesa         493   1.397   1.094  1.091   0.117   4.790
Other            1380  18.615  17.547  6.915   0.124  37.381
Sorrento Valley   369  25.364  24.992  0.798  24.174  27.755

── Which POE is Nearest, by Subarea ──
           subarea nearest_poe_name  count
0      Kearny Mesa        Otay Mesa    206
1          Miramar        Otay Mesa     52
2        Otay Mesa        Otay Mesa    493
3            Other        Otay Mesa   1380
4  Sorrento Valley        Otay Mesa    369


## E2. Indicator 1b — Distance to Nearest Freight Road

Uses `gpd.sjoin_nearest` — matches Christian's method exactly.

In [10]:
# Indicator 1b — Distance to nearest freight road
# Uses gpd.sjoin_nearest — matches Christian's method exactly

nearest_road = gpd.sjoin_nearest(
    businesses,
    major_roads[["geometry"]],
    how="left",
    distance_col="dist_to_freight_road_m"
).drop(columns=["index_right"], errors="ignore")

businesses["dist_to_freight_road_m"]  = nearest_road["dist_to_freight_road_m"]
businesses["dist_to_freight_road_mi"] = businesses["dist_to_freight_road_m"] / METERS_PER_MILE

# Distance category bins — matches Christian's bins exactly
businesses["distance_category"] = pd.cut(
    businesses["dist_to_freight_road_mi"],
    bins=[0, 0.25, 0.5, 1, 2, businesses["dist_to_freight_road_mi"].max()],
    labels=["0-0.25 mi", "0.25-0.5 mi", "0.5-1 mi", "1-2 mi", "2+ mi"],
    include_lowest=True
)

print("── Distance to Freight Road (miles) by Subarea ──")
print(
    businesses.groupby("subarea")["dist_to_freight_road_mi"]
    .agg(n="count", mean="mean", median="median", min="min", max="max")
    .round(3)
)

print("\n── Distance Category Distribution by Subarea ──")
print(
    businesses.groupby(["subarea", "distance_category"]).size()
    .reset_index(name="count")
)



── Distance to Freight Road (miles) by Subarea ──
                    n   mean  median    min    max
subarea                                           
Kearny Mesa       206  0.308   0.270  0.021  0.936
Miramar            52  0.997   0.541  0.118  3.154
Otay Mesa         493  0.321   0.275  0.006  1.189
Other            1380  0.696   0.450  0.020  3.587
Sorrento Valley   369  1.599   1.725  0.041  2.789

── Distance Category Distribution by Subarea ──
            subarea distance_category  count
0       Kearny Mesa         0-0.25 mi     85
1       Kearny Mesa       0.25-0.5 mi    101
2       Kearny Mesa          0.5-1 mi     20
3       Kearny Mesa            1-2 mi      0
4       Kearny Mesa             2+ mi      0
5           Miramar         0-0.25 mi     13
6           Miramar       0.25-0.5 mi     12
7           Miramar          0.5-1 mi      9
8           Miramar            1-2 mi     10
9           Miramar             2+ mi      8
10        Otay Mesa         0-0.25 mi    200
11  

## F. Indicator 2 — Freight Corridor Buffer Overlap

In [11]:
# Indicator 2 — Freight corridor buffer overlap
# Buffer distances in meters; binary inside/outside flag per business point

for col, dist_m in BUFFER_DISTANCES_M.items():
    buf = roads_dissolved.buffer(dist_m)
    businesses[col] = businesses.geometry.within(buf)
    n_in  = int(businesses[col].sum())
    total = len(businesses)
    print(f"  {dist_m}m ({dist_m/METERS_PER_MILE:.2f} mi): "
          f"{n_in}/{total} inside ({100*n_in/total:.1f}%)")

print("\n── Buffer overlap summary by subarea ──")
for col, dist_m in BUFFER_DISTANCES_M.items():
    buf_mi = round(dist_m / METERS_PER_MILE, 2)
    print(f"\n{buf_mi}-mile buffer:")
    summary = (
        businesses.groupby("subarea")
        .agg(
            n_businesses=("dba_name", "count"),
            n_inside=(col, "sum"),
            pct_inside=(col, lambda x: round(100 * x.mean(), 1)),
        )
    )
    print(summary)



  402m (0.25 mi): 733/2500 inside (29.3%)
  805m (0.50 mi): 1374/2500 inside (55.0%)
  1609m (1.00 mi): 1838/2500 inside (73.5%)

── Buffer overlap summary by subarea ──

0.25-mile buffer:
                 n_businesses  n_inside  pct_inside
subarea                                            
Kearny Mesa               206        85        41.3
Miramar                    52        13        25.0
Otay Mesa                 493       200        40.6
Other                    1380       423        30.7
Sorrento Valley           369        12         3.3

0.5-mile buffer:
                 n_businesses  n_inside  pct_inside
subarea                                            
Kearny Mesa               206       186        90.3
Miramar                    52        25        48.1
Otay Mesa                 493       403        81.7
Other                    1380       733        53.1
Sorrento Valley           369        27         7.3

1.0-mile buffer:
                 n_businesses  n_inside  pct_in

## G. Export Results

In [12]:
# Export results

base_cols = [
    "business_acctnum", "dba_name", "industry_group", "naics",
    "address", "city", "state", "zip",
    "latitude", "longitude",
    "CPNAME", "subarea",
    "dist_nearest_poe_m",       "dist_nearest_poe_mi",
    "dist_san_ysidro_m",        "dist_san_ysidro_mi",
    "dist_otay_poe_m",          "dist_otay_poe_mi",
    "nearest_poe_name",
    "dist_to_freight_road_m",   "dist_to_freight_road_mi",
    "distance_category",
]
buf_cols = list(BUFFER_DISTANCES_M.keys())
keep = [c for c in base_cols + buf_cols if c in businesses.columns]

output_gdf = businesses[keep + ["geometry"]].copy()

# CSV (lat/lon preserved for ArcGIS import)
output_gdf.drop(columns=["geometry"]).to_csv("joseph_indicators_output.csv", index=False)
print("Exported: joseph_indicators_output.csv")

# GeoJSON (WGS84 for web/StoryMap)
output_gdf.to_crs(CRS_GEO).to_file("joseph_indicators_output.geojson", driver="GeoJSON")
print("Exported: joseph_indicators_output.geojson")
print(f"Total rows: {len(output_gdf)}")

print("\n── FINAL VALIDATION SUMMARY ──")
val_cols = ["subarea", "dist_nearest_poe_mi"] + buf_cols
print(
    businesses[val_cols].groupby("subarea")
    .agg(["mean", "count"])
    .round(3)
)

print("\nAll done. Load joseph_indicators_output.geojson into ArcGIS for the StoryMap.")



Exported: joseph_indicators_output.csv
Exported: joseph_indicators_output.geojson
Total rows: 2500

── FINAL VALIDATION SUMMARY ──
                dist_nearest_poe_mi       buffer_quarter_mi        \
                               mean count              mean count   
subarea                                                             
Kearny Mesa                  20.545   206             0.413   206   
Miramar                      26.048    52             0.250    52   
Otay Mesa                     1.397   493             0.406   493   
Other                        18.615  1380             0.307  1380   
Sorrento Valley              25.364   369             0.033   369   

                buffer_half_mi       buffer_one_mi        
                          mean count          mean count  
subarea                                                   
Kearny Mesa              0.903   206         1.000   206  
Miramar                  0.481    52         0.654    52  
Otay Mesa            